# Multivariate EDA — New Zealand vs Australia Enrollments

This notebook compares New Zealand and Australian bachelor's-degree enrolments by broad field of study across overlapping years.

**Data sources:**
- NZ: `data/clean/NZ_bachelors_enrollments_2016_2025.csv` — all tertiary providers, FOS_ENR.2 (~77 % university)
- AUS: `data/clean/EnrollmentsAUS_category_with_numeric_key.csv` — universities only, 2016–2024

**Alignment note:** NZ data covers all providers; AUS covers universities only. Raw headcounts are not directly comparable. The analysis uses **indexed change** (base year = 100) and **year-over-year growth rates** as the primary comparison metrics to control for scale differences. Where raw counts are compared, the ~77 % scaling factor is applied to NZ totals.

**Role in the JRGS project:** This notebook assesses whether NZ and AUS share **parallel pre-trends** before the 2021 Job-Ready Graduates Scheme (JRG) — the key identifying assumption for a valid DiD control group. Post-2021 divergence between NZ and AUS (within a field) is potential evidence of a JRG policy effect.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')

# Resolve paths
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
nz_path = aus_path = None

for root in candidate_roots:
    c_nz  = root / 'data' / 'clean' / 'NZ_bachelors_enrollments_2016_2025.csv'
    c_aus = root / 'data' / 'clean' / 'EnrollmentsAUS_category_with_numeric_key.csv'
    if c_nz.exists() and c_aus.exists():
        nz_path, aus_path = c_nz, c_aus
        break

if nz_path is None:
    raise FileNotFoundError(f'Datasets not found from: {Path.cwd()}')

df_nz  = pd.read_csv(nz_path)
df_aus = pd.read_csv(aus_path)

print(f'NZ  shape: {df_nz.shape}  | years: {sorted(df_nz["year"].unique())}')
print(f'AUS shape: {df_aus.shape} | columns: {df_aus.columns.tolist()}')
display(df_nz.head())
display(df_aus.head())

## 1. Harmonize and Merge the Datasets

- AUS is in **wide format** (one column per year); reshape to long.
- NZ is already in **long format** but uses `field_of_study` + `category_key`; AUS uses `Category` + `CategoryKey`.
- Match on **`category_key`** (shared integer key 1–11) to align fields across both countries.
- Keep only years present in both datasets.

In [ ]:
# --- AUS: wide to long ---
aus_yr_cols = [c for c in df_aus.columns if str(c).isdigit()]
aus_long = (df_aus[df_aus['Category'] != 'Total']
            .melt(id_vars=['Category', 'CategoryKey'],
                  value_vars=aus_yr_cols,
                  var_name='year', value_name='Enrollments'))
aus_long['year'] = aus_long['year'].astype(int)
aus_long['Enrollments'] = pd.to_numeric(aus_long['Enrollments'], errors='coerce')
aus_long = aus_long.rename(columns={'Category': 'field_of_study', 'CategoryKey': 'category_key'})
aus_long['country'] = 'Australia'

# --- NZ: filter out aggregate total ---
nz_long = df_nz[df_nz['field_of_study'] != 'Total (all fields)'].copy()
nz_long = nz_long.rename(columns={'total_bachelors': 'Enrollments'})
nz_long['country'] = 'New Zealand'
nz_long = nz_long[['country', 'field_of_study', 'category_key', 'year', 'Enrollments']]

# --- Align on category_key and overlapping years ---
aus_keys  = set(aus_long['category_key'].unique())
nz_keys   = set(nz_long['category_key'].unique())
common_keys = aus_keys & nz_keys

aus_years = set(aus_long['year'].unique())
nz_years  = set(nz_long['year'].unique())
common_years = sorted(aus_years & nz_years)

aus_filt = aus_long[aus_long['category_key'].isin(common_keys) & aus_long['year'].isin(common_years)].copy()
nz_filt  = nz_long[nz_long['category_key'].isin(common_keys) & nz_long['year'].isin(common_years)].copy()

# Harmonise field_of_study label: use AUS label (category_key is shared)
key_to_label = aus_filt.drop_duplicates('category_key').set_index('category_key')['field_of_study'].to_dict()
nz_filt['field_of_study'] = nz_filt['category_key'].map(key_to_label)

combined = pd.concat([aus_filt[['country','field_of_study','category_key','year','Enrollments']],
                      nz_filt[['country','field_of_study','category_key','year','Enrollments']]],
                     ignore_index=True)

print(f'Overlapping years: {common_years}')
print(f'Shared category keys ({len(common_keys)}): {sorted(common_keys)}')
print(f'Combined shape: {combined.shape}')
display(combined.head(12))

## 2. Inspect the Merged Comparison Dataset

Check coverage, summary statistics, and the scale difference between NZ and AUS.

In [ ]:
print('Combined dataset info:')
combined.info()

print('\nSummary by country:')
display(combined.groupby('country')['Enrollments'].describe().round(0))

# Scale ratio: AUS/NZ by field in the latest common year
latest = max(common_years)
wide_latest = (combined[combined['year'] == latest]
               .pivot_table(index='field_of_study', columns='country', values='Enrollments')
               .dropna())
wide_latest['AUS/NZ ratio'] = (wide_latest['Australia'] / wide_latest['New Zealand']).round(2)
print(f'\nScale ratio AUS/NZ in {latest}:')
print(wide_latest.sort_values('AUS/NZ ratio', ascending=False).to_string())
print(f'\nNote: NZ data includes all providers (~77% university). True NZ university-only AUS/NZ ratio is approximately AUS/(NZ*0.77).')

## 3. Compare Overall and Category-Level Trends

Raw totals, indexed change (base 2019 = 100), and YoY growth — indexed views are the most informative given the provider-coverage difference.

In [ ]:
# --- Indexed trend: base 2019 = 100 (pre-JRG baseline) ---
BASE_YEAR = 2019

totals = combined.groupby(['year','country'])['Enrollments'].sum().reset_index()
base_totals = totals[totals['year'] == BASE_YEAR].set_index('country')['Enrollments']
totals['Index'] = totals.apply(lambda r: r['Enrollments'] / base_totals[r['country']] * 100, axis=1)
totals['YoY_%'] = totals.groupby('country')['Enrollments'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=totals, x='year', y='Index', hue='country', marker='o', linewidth=2.5, ax=axes[0])
axes[0].axvline(2021, color='red', ls='--', lw=1.2, label='JRG (AUS)')
axes[0].axhline(100, color='grey', ls=':', lw=1)
axes[0].set_title(f'Total Enrolments Indexed ({BASE_YEAR}=100)')
axes[0].set_ylabel('Index')
axes[0].legend()

sns.lineplot(data=totals.dropna(subset=['YoY_%']), x='year', y='YoY_%',
             hue='country', marker='o', linewidth=2.5, ax=axes[1])
axes[1].axvline(2021, color='red', ls='--', lw=1.2)
axes[1].axhline(0, color='grey', ls=':', lw=1)
axes[1].set_title('Year-over-Year Enrolment Change (%)')
axes[1].set_ylabel('YoY % change')
plt.tight_layout()
plt.show()
display(totals)

# --- Facet: indexed by field ---
base_vals = (combined[combined['year'] == BASE_YEAR]
             .groupby(['country','field_of_study'])['Enrollments'].sum()
             .reset_index().rename(columns={'Enrollments': 'Base'}))
indexed = combined.merge(base_vals, on=['country','field_of_study'], how='left')
indexed['Index'] = indexed['Enrollments'] / indexed['Base'] * 100

g = sns.relplot(data=indexed.dropna(subset=['Index']),
                x='year', y='Index', hue='country', col='field_of_study',
                col_wrap=3, kind='line', marker='o',
                facet_kws={'sharey': False, 'sharex': True},
                height=3.2, aspect=1.2)
g.set_titles('{col_name}')
for ax in g.axes.flat:
    ax.axvline(2021, color='red', ls='--', lw=1)
    ax.axhline(100, color='grey', ls=':', lw=0.8)
    ax.tick_params(axis='x', rotation=45)
g.fig.suptitle(f'Indexed Enrolment Trends by Field ({BASE_YEAR}=100)', y=1.02)
plt.show()

## 4. Multivariate Relationship Analysis

Scatter plots, correlation, and difference heatmaps.

In [ ]:
# Wide format for scatter
wide = (combined.pivot_table(index=['field_of_study','year'], columns='country', values='Enrollments')
        .reset_index().dropna())
wide.columns.name = None
wide['Gap (AUS-NZ)'] = wide['Australia'] - wide['New Zealand']
wide['Ratio AUS/NZ'] = (wide['Australia'] / wide['New Zealand']).round(3)

plt.figure(figsize=(9, 7))
sns.scatterplot(data=wide, x='New Zealand', y='Australia', hue='field_of_study', style='year', s=90)
sns.regplot(data=wide, x='New Zealand', y='Australia', scatter=False,
            color='black', line_kws={'linestyle': '--'})
plt.title('NZ vs AUS Enrolments by Field-Year')
plt.xlabel('New Zealand Enrolments (all providers)')
plt.ylabel('Australia Enrolments (universities)')
plt.tight_layout()
plt.show()

# Category-level correlation (NZ vs AUS within field)
corr_by_field = (wide.groupby('field_of_study')
                 .apply(lambda g: g['New Zealand'].corr(g['Australia']) if len(g) > 2 else np.nan)
                 .reset_index(name='Pearson r')
                 .sort_values('Pearson r', ascending=False))
print('Field-level NZ–AUS correlation:')
display(corr_by_field)

# Difference heatmap
diff_heatmap = wide.pivot_table(index='field_of_study', columns='year', values='Gap (AUS-NZ)')
plt.figure(figsize=(12, 7))
sns.heatmap(diff_heatmap, annot=True, fmt='.0f', cmap='coolwarm', center=0)
plt.title('Enrollment Gap (AUS - NZ) by Field and Year')
plt.tight_layout()
plt.show()

## 5. Distribution Comparison (Tukey Box Plots)

Compare enrolment distributions for each field across the two countries.

In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(data=combined, x='Enrollments', y='field_of_study', hue='country', whis=1.5)
sns.stripplot(data=combined, x='Enrollments', y='field_of_study', hue='country',
              dodge=True, alpha=0.45, size=3,
              palette={'Australia': 'black', 'New Zealand': 'grey'})

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title='Country', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.title('Tukey Box Plot: NZ vs AUS Enrolments by Field')
plt.xlabel('Enrolments')
plt.tight_layout()
plt.show()

## 6. Parallel Trends Assessment

The core DiD identifying assumption: **before 2021 (pre-JRG), NZ and AUS enrolment trends should move in parallel within each field**. This section tests that assumption and flags fields where it may be violated.

> A **parallel trend** does not require identical levels — it requires that the *growth rates* (or log changes) in NZ and AUS track together during the pre-treatment period (2016–2020).

In [ ]:
PRE_YEARS  = [y for y in common_years if y < 2021]
POST_YEARS = [y for y in common_years if y >= 2021]

# Log-indexed trend: pre-period only (parallel trends visualisation)
pre_data = indexed[indexed['year'].isin(PRE_YEARS)].dropna(subset=['Index'])

g = sns.relplot(data=pre_data,
                x='year', y='Index', hue='country', col='field_of_study',
                col_wrap=3, kind='line', marker='o',
                facet_kws={'sharey': False, 'sharex': True},
                height=3.2, aspect=1.2)
g.set_titles('{col_name}')
for ax in g.axes.flat:
    ax.axhline(100, color='grey', ls=':', lw=0.8)
    ax.tick_params(axis='x', rotation=45)
g.fig.suptitle(f'Pre-Treatment Trends ({BASE_YEAR}=100) — Parallel Trends Check', y=1.02)
plt.show()

# YoY % change by country — pre-period
pre_yoy = (combined[combined['year'].isin(PRE_YEARS + [min(PRE_YEARS) - 1])]
           .sort_values(['country','field_of_study','year'])
           .copy())
pre_yoy['YoY_%'] = pre_yoy.groupby(['country','field_of_study'])['Enrollments'].pct_change() * 100
pre_yoy = pre_yoy.dropna(subset=['YoY_%'])

# Correlation of YoY growth rates between NZ and AUS by field
from scipy import stats as scipy_stats
trend_corr = []
for field, g in pre_yoy.groupby('field_of_study'):
    aus_g = g[g['country'] == 'Australia']['YoY_%'].values
    nz_g  = g[g['country'] == 'New Zealand']['YoY_%'].values
    if len(aus_g) >= 3 and len(nz_g) >= 3 and len(aus_g) == len(nz_g):
        r, p = scipy_stats.pearsonr(aus_g, nz_g)
        trend_corr.append({'field_of_study': field, 'r_YoY_growth': round(r, 3), 'p': round(p, 4)})

trend_corr_df = pd.DataFrame(trend_corr).sort_values('r_YoY_growth', ascending=False)
print('Pre-treatment YoY growth rate correlation (NZ vs AUS) — parallel trends evidence:')
display(trend_corr_df)
print('\nFields with r < 0.5 may have weaker pre-treatment parallel trends.')

## 7. Additional Statistical Checks

Correlation, nonlinearity, heterogeneity, and Simpson's paradox check.

In [ ]:
try:
    from scipy import stats
except ImportError:
    stats = None

# Overall NZ–AUS correlation
if stats and not wide.empty:
    r_overall, p_overall = stats.pearsonr(wide['New Zealand'], wide['Australia'])
    print(f'Overall r(NZ, AUS) = {r_overall:.3f}  (p={p_overall:.4f})')

# Nonlinearity: linear vs quadratic
x = wide['New Zealand'].to_numpy(dtype=float)
y = wide['Australia'].to_numpy(dtype=float)
sort_idx = np.argsort(x)
x_s, y_s = x[sort_idx], y[sort_idx]

lin_c  = np.polyfit(x_s, y_s, 1)
quad_c = np.polyfit(x_s, y_s, 2)
lin_p  = np.polyval(lin_c, x_s)
quad_p = np.polyval(quad_c, x_s)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

print(f'Linear fit R²: {r2(y_s, lin_p):.4f}  |  Quadratic fit R²: {r2(y_s, quad_p):.4f}')

plt.figure(figsize=(9, 6))
plt.scatter(x_s, y_s, color='black', alpha=0.6, label='Observed')
plt.plot(x_s, lin_p, color='steelblue', label='Linear fit')
plt.plot(x_s, quad_p, color='red', ls='--', label='Quadratic fit')
plt.title('Nonlinearity Check: NZ vs AUS Enrolments')
plt.xlabel('New Zealand Enrolments')
plt.ylabel('Australia Enrolments')
plt.legend()
plt.tight_layout()
plt.show()

# Gap heterogeneity across fields
gap_groups = [g['Gap (AUS-NZ)'].values for _, g in wide.groupby('field_of_study') if len(g) > 1]
if stats and len(gap_groups) >= 2:
    lev_stat, lev_p = stats.levene(*gap_groups, center='median')
    f_stat,   f_p   = stats.f_oneway(*gap_groups)
    print(f"Levene's test on AUS-NZ gap: F={lev_stat:.3f}, p={lev_p:.4g}")
    print(f'ANOVA on AUS-NZ gap: F={f_stat:.3f}, p={f_p:.4g}')

## Data Characteristics & First-Order Effects

**Variables:** Merged panel of NZ and AUS enrolment headcounts at field × year level. NZ data includes all tertiary providers (not universities only); AUS is universities only. The ~77 % university share in NZ (from 2025 FOS_ENR.3) is field-heterogeneous, so per-field scaling will differ.

**Key DiD concept:** The counterfactual for AUS under no-JRG is the growth path NZ actually experienced. Valid DiD requires that AUS and NZ would have followed the same trajectory absent the policy — assessed via parallel pre-trends (Section 6). Post-2021 **divergence** is the treatment effect estimate.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid')

# Reload clean combined data (self-contained cell)
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
clean_dir = next((r / 'data' / 'clean' for r in candidate_roots if (r / 'data' / 'clean').exists()), None)
assert clean_dir, 'Cannot find data/clean directory'

aus = pd.read_csv(clean_dir / 'EnrollmentsAUS_category_with_numeric_key.csv')
nz  = pd.read_csv(clean_dir / 'NZ_bachelors_enrollments_2016_2025.csv')

aus_yr_cols = [c for c in aus.columns if str(c).isdigit()]
aus_l = (aus[aus['Category'] != 'Total']
         .melt(id_vars=['Category','CategoryKey'], value_vars=aus_yr_cols, var_name='year', value_name='AUS'))
aus_l['year'] = aus_l['year'].astype(int)
nz_l  = nz[nz['field_of_study'] != 'Total (all fields)'].rename(
    columns={'total_bachelors': 'NZ', 'field_of_study': 'Category', 'category_key': 'CategoryKey'})

merged = aus_l.merge(nz_l[['CategoryKey','year','NZ']], on=['CategoryKey','year'], how='inner').dropna()

print('=== NZ–AUS Multivariate — Variable Summary ===')
print(f'Merged shape: {merged.shape}')
print(f'Years: {sorted(merged["year"].unique())}')
r_overall, p = stats.pearsonr(merged['NZ'], merged['AUS'])
print(f'Overall r(NZ, AUS) = {r_overall:.3f}  (p={p:.4f})')

# Simpson's paradox check
cat_rs = []
for cat, g in merged.groupby('Category'):
    if len(g) >= 3:
        r, p = stats.pearsonr(g['NZ'], g['AUS'])
        cat_rs.append({'Category': cat, 'r': r})
cat_rs_df = pd.DataFrame(cat_rs).set_index('Category')
neg = (cat_rs_df['r'] < 0).sum()
if neg > 0 and r_overall > 0:
    print(f"Simpson's Paradox: {neg} field(s) show negative NZ–AUS correlation despite positive aggregate r")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('NZ–AUS Enrolments — Data Characteristics', fontsize=12, fontweight='bold')

colors = ['#e74c3c' if r < 0 else '#27ae60' for r in cat_rs_df['r']]
axes[0].barh(cat_rs_df.index, cat_rs_df['r'], color=colors)
axes[0].axvline(r_overall, color='navy', ls='--', lw=2, label=f'Aggregate r={r_overall:.2f}')
axes[0].set_title('A. Per-field NZ–AUS correlation')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='y', labelsize=8)

sc = axes[1].scatter(merged['NZ'], merged['AUS'], c=merged['year'], cmap='viridis', alpha=0.8, s=50)
plt.colorbar(sc, ax=axes[1], label='Year')
axes[1].set_xlabel('NZ Enrolments')
axes[1].set_ylabel('AUS Enrolments')
axes[1].set_title('B. NZ vs AUS scatter (colour=year)')

axes[2].hist(np.log1p(merged['NZ']),  bins=20, color='steelblue',  edgecolor='white', alpha=0.7, label='NZ')
axes[2].hist(np.log1p(merged['AUS']), bins=20, color='darkorange', edgecolor='white', alpha=0.7, label='AUS')
axes[2].set_xlabel('log(1 + Enrolments)')
axes[2].set_title('C. Log-transformed distributions')
axes[2].legend()

plt.tight_layout()
plt.show()

### What Is Learned

1. **Variable characteristics:** TODO — merged panel with NZ (all providers) and AUS (universities). Scale differs ~4–6× depending on field; indexed comparisons are preferred.
2. **Data cleaning:** TODO — note which fields were dropped if coverage was incomplete.
3. **Aggregate correlation:** TODO — if r(NZ,AUS) is high overall, scale artefact likely; field-level correlations are more informative.
4. **Parallel pre-trends:** TODO — key finding. List which fields pass and which fail the parallel trends assessment.
5. **Modelling implications:** TODO — fields with weak pre-trends may warrant sensitivity checks in the DiD regressions. NZ is the second control group alongside the UK; consistency of NZ and UK pre-trends strengthens the DiD design.